# Fig 4A / 4B / S9A (v2) — Meta-analysis of similar populations, ADSP + UKB (data processing)

v2 design (`32.Fig4A_4B_S9A.ADSP_UKB_Meta_v2`): total n = 10,000, NCP = n × per-SNP h2g, ADSP share 0–100%, in-sample LD from the sampled rows, references from the whole panels, and SuSiE-rRSS fits next to SuSiE-RSS. The v1 notebook and its `figure_data` are left as they are; everything here is written with a `_v2` name. The identity-LD fits are not used.

**Panel sets** (keys of `pal_meta_adsp_ukb`; the estimated mixture is shown as "Estimated LD mixture" by `meta_ld_label()`):

| | Fig 4 (`main`) | Supplement (`rrss`) |
|---|---|---|
| Estimated mixture | SuSiE-RSS, weights estimated then fixed (`manual_refpanel`, as v1) | SuSiE-rRSS (`mix_refpanel_rrss`) |
| Oracle | SuSiE-RSS on the in-sample LD (`gen_insample`) | same |
| ADSP only / UKB only | SuSiE-RSS (`ADSP_refpanel`, `UKB_refpanel`) | SuSiE-rRSS (`*_rrss`) |

The Oracle uses the LD of the sampled rows themselves, the exact generative LD of z_meta. At ADSP share 100% all 10,000 ADSP sketch rows are sampled, so the in-sample LD is the ADSP panel LD and the Oracle, ADSP-only and true-weight fits must agree (checked below).

**SuSiE-rRSS is the final result:** for every `*_rrss` fit, the SER fallback model replaces the fit when `R_reliability_flag` is TRUE (the `<fit>_SER` rows of the summary).

**Inputs:** `ADSP_UKB_meta_v2/summary/` (`[meta_v2_summary]`: the summary table, `_dropped`, `_ld_diff`)

**Outputs:**
- `figure_data/Fig4A_v2.ADSP_UKB_Weight_Estimation` — estimated vs. true ADSP proportion: SuSiE-RSS mixture (Fig 4) and SuSiE-rRSS mixture (supplement) (+ `_accuracy.csv`)
- `figure_data/Fig4B_v2.ADSP_UKB_Mix_Power_FDR` — power / FDR (bootstrap 95% CI) of the two panel sets: `panel` = `main` (SuSiE-RSS) or `rrss` (SuSiE-rRSS); every fit in `Fig4B_v2.ADSP_UKB_Mix_Power_FDR_All_Fits`
- `figure_data/FigS9_v2.ADSP_UKB_SER_Fallback` — SER fallback rate of each SuSiE-rRSS fit
- `figure_data/FigS9_v2.ADSP_UKB_LD_Difference` — mean LD difference of each reference from the in-sample LD

In [15]:
source("../../R/utils.R")

sum_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_meta_v2/summary"
res_file <- list.files(sum_dir, pattern = "_meta_v2_summary\\.rds$", full.names = TRUE)
stopifnot(length(res_file) == 1)
n_boot <- 1000
not_used <- c("identity", "identity_rrss")      # identity-LD fits

method_labels <- c(
  gen_insample       = "Oracle: in-sample LD (SuSiE-RSS)",
  gen_refpanel       = "Reference mixture, true weight (SuSiE-RSS)",
  manual_refpanel    = "Estimated mixture (SuSiE-RSS)",
  mix_refpanel       = "Estimated mixture, weight fitted jointly (SuSiE-RSS)",
  ADSP_refpanel      = "ADSP only (SuSiE-RSS)",
  UKB_refpanel       = "UKB only (SuSiE-RSS)",
  mix_refpanel_rrss  = "Estimated mixture (SuSiE-rRSS)",
  ADSP_refpanel_rrss = "ADSP only (SuSiE-rRSS)",
  UKB_refpanel_rrss  = "UKB only (SuSiE-rRSS)"
)
## panel sets: values are keys of pal_meta_adsp_ukb
main_labels <- c(
  manual_refpanel = "SuSiE-rRSS",      # palette key of the estimated mixture (shown as "Estimated LD mixture")
  gen_insample    = "Oracle",
  ADSP_refpanel   = "ADSP only",
  UKB_refpanel    = "UKB only"
)
rrss_labels <- c(
  mix_refpanel_rrss  = "SuSiE-rRSS",
  gen_insample       = "Oracle",
  ADSP_refpanel_rrss = "ADSP only",
  UKB_refpanel_rrss  = "UKB only"
)
stopifnot(setequal(main_labels, names(pal_meta_adsp_ukb)), setequal(rrss_labels, names(pal_meta_adsp_ukb)))
panels <- bind_rows(tibble(panel = "main", method = names(main_labels), label = unname(main_labels)),
                    tibble(panel = "rrss", method = names(rrss_labels), label = unname(rrss_labels)))

## bootstrap 95% CI of power / FDR over simulations; same estimates and resampling as
## boot_cell_mix() (R/utils.R), vectorized over the B resamples
boot_cell_fast <- function(d, B = 1000) {
  R   <- nrow(d)
  idx <- matrix(sample.int(R, R * B, replace = TRUE), nrow = R)
  bs  <- function(x) colSums(matrix(x[idx], nrow = R))
  tv <- bs(d$true_var_number); av <- bs(d$total_var_number)
  ts <- bs(d$true_set_number); as <- bs(d$total_set_number)
  fdn <- sum(d$total_set_number)
  qp <- .q(tv / av); qf <- .q(ifelse(as == 0, NA_real_, 1 - ts / as))
  tibble(R = R,
         power = sum(d$true_var_number) / sum(d$total_var_number), power_lo = qp[1], power_hi = qp[2],
         FDR   = if (fdn == 0) 0 else 1 - sum(d$true_set_number) / fdn, FDR_lo = qf[1], FDR_hi = qf[2])
}

## Simulations and dropped runs

In [16]:
res     <- readRDS(res_file) %>% filter(!str_remove(method, "_SER$") %in% not_used)
dropped <- readRDS(sub("\\.rds$", "_dropped.rds", res_file))

## simulations per setting, and the dropped ones (causal variant monomorphic in a sampled cohort, ...)
res %>% distinct(input_file, ncp, causal, prop) %>% count(ncp, causal, prop) %>%
  pivot_wider(names_from = prop, values_from = n, names_prefix = "ADSP share ")
## an empty table (no columns) when no simulation was dropped
if (nrow(dropped)) count(dropped, reason, sort = TRUE) else cat("No simulation dropped\n")
res %>% filter(!fit_ok) %>% count(method, name = "n_failed_fits")
stopifnot(all(str_remove(unique(res$method), "_SER$") %in% names(method_labels)))

ncp,causal,ADSP share 0.0,ADSP share 0.2,ADSP share 0.5,ADSP share 0.8,ADSP share 1.0
<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>
80,1,496,496,496,496,496
80,2,496,479,373,469,496
80,3,496,466,344,442,496
120,1,496,496,496,496,496
120,2,496,488,385,474,496
120,3,496,474,341,418,496
250,1,496,496,496,496,496
250,2,496,483,367,457,496
250,3,496,471,318,427,496


No simulation dropped


method,n_failed_fits
<chr>,<int>
mix_refpanel,18
mix_refpanel_rrss,23


## Final SuSiE-rRSS result and SER fallback

In [17]:
## final SuSiE-rRSS result: the SER model replaces a *_rrss fit when it is flagged
base <- res %>% filter(!str_ends(method, "_SER"))
ser  <- res %>% filter(str_ends(method, "_SER")) %>% mutate(method = str_remove(method, "_SER$"))
use_ser <- base %>%
  filter(str_ends(method, "_rrss"), R_reliability_flag %in% TRUE) %>%
  select(input_file, method) %>%
  semi_join(x = ser, by = c("input_file", "method"))
final <- base %>%
  anti_join(select(use_ser, input_file, method), by = c("input_file", "method")) %>%
  bind_rows(mutate(use_ser, ser_fallback = TRUE)) %>%
  mutate(ser_fallback = ser_fallback %in% TRUE,
         rrss         = str_ends(method, "_rrss"),
         weight       = as.numeric(prop),
         method_label = factor(unname(method_labels[method]), levels = unname(method_labels))) %>%
  filter(fit_ok)
stopifnot(!any(duplicated(select(final, input_file, method))))

## flagged *_rrss fits without a stored SER model keep the fit (should be none)
base %>% filter(str_ends(method, "_rrss"), R_reliability_flag %in% TRUE) %>%
  anti_join(use_ser, by = c("input_file", "method")) %>% count(method, name = "flagged_without_SER")

## SER fallback rate of each SuSiE-rRSS fit
fallback <- final %>% filter(rrss) %>%
  group_by(method, method_label, ncp, causal, weight) %>%
  summarise(n_sim = n(), n_flagged = sum(R_reliability_flag %in% TRUE),
            pct_fallback = 100 * mean(ser_fallback), .groups = "drop") %>%
  left_join(filter(panels, panel == "rrss") %>% select(method, label), by = "method")
fallback %>% group_by(method_label, ncp) %>%
  summarise(pct_fallback = round(100 * sum(n_flagged) / sum(n_sim), 1), .groups = "drop") %>%
  pivot_wider(names_from = ncp, values_from = pct_fallback, names_prefix = "NCP ")
save_data(fallback, "FigS9_v2.ADSP_UKB_SER_Fallback")

## at ADSP share 100% the in-sample LD is the ADSP panel LD (all sketch rows sampled), so the Oracle,
## ADSP-only and true-weight SuSiE-RSS fits use the same LD and must give the same credible sets
same_fit <- final %>%
  filter(weight == 1, method %in% c("gen_insample", "ADSP_refpanel", "gen_refpanel")) %>%
  select(input_file, method, total_set_number, true_set_number, true_var_number, total_set_length) %>%
  pivot_wider(names_from = method, values_from = c(total_set_number, true_set_number, true_var_number, total_set_length))
tibble(n_sim = nrow(same_fit),
       oracle_eq_ADSP_only = sum(same_fit$total_set_number_gen_insample == same_fit$total_set_number_ADSP_refpanel &
                                 same_fit$true_set_number_gen_insample  == same_fit$true_set_number_ADSP_refpanel &
                                 same_fit$total_set_length_gen_insample == same_fit$total_set_length_ADSP_refpanel, na.rm = TRUE),
       oracle_eq_true_weight = sum(same_fit$total_set_number_gen_insample == same_fit$total_set_number_gen_refpanel &
                                   same_fit$true_set_number_gen_insample  == same_fit$true_set_number_gen_refpanel &
                                   same_fit$total_set_length_gen_insample == same_fit$total_set_length_gen_refpanel, na.rm = TRUE))

method,flagged_without_SER
<chr>,<int>


method_label,NCP 80,NCP 120,NCP 250,NCP 500
<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Estimated mixture (SuSiE-rRSS),0.0,0.1,0.6,1.5
ADSP only (SuSiE-rRSS),0.4,0.6,2.1,9.4
UKB only (SuSiE-rRSS),0.4,0.6,1.8,7.7


n_sim,oracle_eq_ADSP_only,oracle_eq_true_weight
<int>,<int>,<int>
5952,5952,5952


## Fig 4A (v2): weight estimation

In [18]:
## one row per simulation: estimated ADSP weight (omega[1]) of the SuSiE-RSS (Fig 4) and SuSiE-rRSS (supplement) mixtures
om <- res %>%
  filter(method == "mix_refpanel") %>%
  distinct(input_file, ncp, causal, prop, w, omega_1, omega_rrss_1) %>%
  pivot_longer(c(omega_1, omega_rrss_1), names_to = "model", values_to = "omega") %>%
  filter(!is.na(omega)) %>%
  mutate(model  = factor(recode(model, omega_1 = "SuSiE-RSS", omega_rrss_1 = "SuSiE-rRSS"), levels = c("SuSiE-RSS", "SuSiE-rRSS")),
         weight = as.numeric(prop))
stopifnot(all(abs(om$w - om$weight) < 1e-6))      # realized share = design share

om_acc <- om %>%
  group_by(model, ncp, causal, weight) %>%
  summarise(mean_omega = mean(omega), sd_omega = sd(omega),
            MAE = mean(abs(omega - weight)), n = n(), .groups = "drop")
om_acc %>% group_by(model, weight) %>%
  summarise(MAE = round(mean(MAE), 3), n = sum(n), .groups = "drop") %>%
  pivot_wider(names_from = model, values_from = c(MAE, n))
write_csv(om_acc, file.path(data_dir, "Fig4A_v2.ADSP_UKB_Weight_Estimation_accuracy.csv"))
save_data(om, "Fig4A_v2.ADSP_UKB_Weight_Estimation")

weight,MAE_SuSiE-RSS,MAE_SuSiE-rRSS,n_SuSiE-RSS,n_SuSiE-rRSS
<dbl>,<dbl>,<dbl>,<int>,<int>
0.0,0.002,0.001,5952,5952
0.2,0.051,0.051,5762,5761
0.5,0.027,0.027,4791,4785
0.8,0.056,0.056,5520,5522
1.0,0.003,0.003,5952,5952


## Fig 4B (v2): power / FDR

In [19]:
## bootstrap 95% CIs for every fit x setting
set.seed(1)
summary_ci <- final %>%
  group_by(method, method_label, rrss, ncp, causal, weight) %>%
  group_modify(~ boot_cell_fast(.x, B = n_boot)) %>%
  ungroup()

w_levels <- sort(unique(summary_ci$weight))
to_long <- function(df) df %>%
  pivot_longer(c(power, power_lo, power_hi, FDR, FDR_lo, FDR_hi), names_to = "key", values_to = "value") %>%
  mutate(metric = factor(if_else(str_starts(key, "power"), "Power", "FDR"), levels = c("Power", "FDR")),
         key    = case_when(str_ends(key, "_lo") ~ "lo", str_ends(key, "_hi") ~ "hi", TRUE ~ "est")) %>%
  pivot_wider(names_from = key, values_from = value) %>%
  mutate(ncp_num = ncp,
         ncp     = fct_reorder(paste0("NCP = ", ncp_num), ncp_num),
         causal  = factor(paste0(causal, " causal"), levels = paste0(sort(unique(as.integer(causal))), " causal")),
         weight  = factor(weight, levels = w_levels, labels = paste0(round(w_levels * 100), "%")))

all_fits <- to_long(summary_ci)
## the two panel sets (the Oracle is in both)
pl <- summary_ci %>% inner_join(panels, by = "method", relationship = "many-to-many") %>% to_long() %>%
  mutate(method_key = method, method = factor(label, levels = names(pal_meta_adsp_ukb)), panel = factor(panel, levels = c("main", "rrss")))
stopifnot(!any(is.na(pl$method)))

## panel sets averaged over settings (quick check)
pl %>% group_by(panel, method, metric) %>% summarise(mean_est = round(mean(est, na.rm = TRUE), 3), .groups = "drop") %>%
  pivot_wider(names_from = metric, values_from = mean_est)
save_data(pl, "Fig4B_v2.ADSP_UKB_Mix_Power_FDR")
save_data(all_fits, "Fig4B_v2.ADSP_UKB_Mix_Power_FDR_All_Fits")

panel,method,Power,FDR
<fct>,<fct>,<dbl>,<dbl>
main,SuSiE-rRSS,0.979,0.027
main,Oracle,0.979,0.025
main,ADSP only,0.975,0.049
main,UKB only,0.975,0.048
rrss,SuSiE-rRSS,0.970,0.021
rrss,Oracle,0.979,0.025
rrss,ADSP only,0.954,0.024
rrss,UKB only,0.955,0.026


## LD difference from the in-sample LD

In [20]:
## LD difference of each reference from the in-sample LD, per simulation; the estimated mixtures depend on z, so NCP is kept
ld_diff <- readRDS(sub("\\.rds$", "_ld_diff.rds", res_file)) %>% filter(!reference %in% not_used)
ld_metrics <- intersect(c("mse", "max_abs_diff", "concord"), names(ld_diff))
ld_summ <- ld_diff %>%
  mutate(weight = as.numeric(prop)) %>%
  group_by(across(any_of(c("reference", "pairs"))), ncp, weight) %>%
  summarise(n_sim = n(), across(all_of(ld_metrics), ~ mean(.x, na.rm = TRUE)), .groups = "drop")
ld_summ %>% group_by(across(any_of(c("reference", "pairs")))) %>%
  summarise(across(all_of(ld_metrics), ~ signif(mean(.x, na.rm = TRUE), 3)), .groups = "drop")
save_data(ld_summ, "FigS9_v2.ADSP_UKB_LD_Difference")

reference,pairs,mse,max_abs_diff,concord
<chr>,<chr>,<dbl>,<dbl>,<dbl>
ADSP_refpanel,all,3.30e-04,0.1230,0.996
ADSP_refpanel,insample_r_gt_0.05,3.50e-04,0.1200,0.998
UKB_refpanel,all,3.32e-04,0.1300,0.996
UKB_refpanel,insample_r_gt_0.05,3.52e-04,0.1260,0.998
gen_refpanel,all,3.95e-05,0.0254,0.999
gen_refpanel,insample_r_gt_0.05,3.62e-05,0.0241,1.000
mix_refpanel,all,7.70e-05,0.0274,0.999
mix_refpanel,insample_r_gt_0.05,7.45e-05,0.0262,1.000
mix_refpanel_rrss,all,4.09e-05,0.0267,0.999
